### Allscripts Sunrise (SCM) - Care Site Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3location

**Strategy:**
- Extract location/facility data from dbo_cv3location
- Map TypeCode to place of service concepts
- Use GUID as care_site_source_value
- Populate care_site_name from Name field
- Filter to Active locations only

**Note:**
Care site is a reference table - loaded once, rarely changes

# Transformation

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_care_site AS
SELECT DISTINCT
  loc.Name AS care_site_name,
  COALESCE(pos_concept.omop_concept_id, 0) AS place_of_service_concept_id,
  NULL AS location_id,
  CONCAT_WS(CHAR(31),'allscripts_scm',  'dbo_cv3location', 'id', CAST(loc.GUID AS BIGINT)) AS care_site_source_value,
  loc.Code AS place_of_service_source_value,
  'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3location loc
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept pos_concept
  ON pos_concept.source_id = CAST(loc.PlaceOfServiceID AS STRING)
  AND pos_concept.domain_id = 'Place of Service'
  AND pos_concept.source_system = 'allscripts_scm'
WHERE loc.Active = TRUE
  AND loc.Name IS NOT NULL

# Merge to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.care_site AS t
USING (
    SELECT DISTINCT *
    FROM silver_care_site
) AS s
ON t.care_site_source_value = s.care_site_source_value

WHEN MATCHED AND NOT (
       t.care_site_name <=> s.care_site_name
   AND t.place_of_service_concept_id <=> s.place_of_service_concept_id
   AND t.location_id <=> s.location_id
   AND t.place_of_service_source_value <=> s.place_of_service_source_value
   AND t.source_system <=> s.source_system
)
THEN UPDATE SET
    t.care_site_name = s.care_site_name,
    t.place_of_service_concept_id = s.place_of_service_concept_id,
    t.location_id = s.location_id,
    t.place_of_service_source_value = s.place_of_service_source_value,
    t.source_system = s.source_system,
    t.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    care_site_name,
    place_of_service_concept_id,
    location_id,
    care_site_source_value,
    place_of_service_source_value,
    source_system,
    last_mod_tsp
)
VALUES (
    s.care_site_name,
    s.place_of_service_concept_id,
    s.location_id,
    s.care_site_source_value,
    s.place_of_service_source_value,
    s.source_system,
    CURRENT_TIMESTAMP()
);

# Populate Mapping Table

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_care_site (
    source_system,
    care_site_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT DISTINCT
    cs.source_system,
    cs.care_site_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(cs.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM _exponent.omop_silver.care_site cs
WHERE cs.care_site_source_value NOT IN (
    SELECT
        m.care_site_source_value
    FROM _exponent.omop_mapping.source_to_care_site m
    WHERE m.active_flag = TRUE
);

# Merge to Gold

In [0]:
%sql
MERGE INTO _exponent.omop.care_site AS gold
USING (
  SELECT 
    stc.care_site_id,
    s.care_site_name,
    s.place_of_service_concept_id,
    s.location_id,
    s.care_site_source_value,
    s.place_of_service_source_value
  FROM _exponent.omop_silver.care_site s
  JOIN _exponent.omop_mapping.source_to_care_site stc
    ON stc.care_site_source_value = s.care_site_source_value
   AND stc.active_flag = TRUE
) AS src
ON gold.care_site_id = src.care_site_id

WHEN MATCHED AND NOT (
       gold.care_site_name <=> src.care_site_name
   AND gold.place_of_service_concept_id <=> src.place_of_service_concept_id
   AND gold.location_id <=> src.location_id
   AND gold.care_site_source_value <=> src.care_site_source_value
   AND gold.place_of_service_source_value <=> src.place_of_service_source_value
)
THEN UPDATE SET
  gold.care_site_name = src.care_site_name,
  gold.place_of_service_concept_id = src.place_of_service_concept_id,
  gold.location_id = src.location_id,
  gold.care_site_source_value = src.care_site_source_value,
  gold.place_of_service_source_value = src.place_of_service_source_value

WHEN NOT MATCHED THEN INSERT (
  care_site_id,
  care_site_name,
  place_of_service_concept_id,
  location_id,
  care_site_source_value,
  place_of_service_source_value
)
VALUES (
  src.care_site_id,
  src.care_site_name,
  src.place_of_service_concept_id,
  src.location_id,
  src.care_site_source_value,
  src.place_of_service_source_value
);